#Listing B-1: TensorFlow/Keras Version of Listing 4-1
This code is a TensorFlow/Keras version of the simple model shown in Listing 4-1.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# --------------------------------------------------
# 1. Load and prepare data
# --------------------------------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to [0, 1] range
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Apply same normalization as PyTorch version (mean=0.1307, std=0.3081)
x_train = (x_train - 0.1307) / 0.3081
x_test = (x_test - 0.1307) / 0.3081

# Flatten images from 28x28 to 784
x_train = x_train.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)

# --------------------------------------------------
# 2. Define or load model
# --------------------------------------------------
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(28 * 28,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10)
])

# --------------------------------------------------
# 3. Specify loss function and optimizer
# --------------------------------------------------
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# --------------------------------------------------
# 4. Train model
# --------------------------------------------------
history = model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=5,
    validation_data=(x_test, y_test),
    verbose=0
)

# --------------------------------------------------
# 5. Validate model during training
# --------------------------------------------------
for epoch in range(5):
    train_loss = history.history['loss'][epoch]
    val_loss = history.history['val_loss'][epoch]
    val_accuracy = history.history['val_accuracy'][epoch] * 100

    print(
        f"Epoch {epoch + 1}: "
        f"train loss={train_loss:.4f}, "
        f"val loss={val_loss:.4f}, "
        f"val acc={val_accuracy:.2f}%"
    )

# --------------------------------------------------
# 6. Evaluate and test (final pass)
# --------------------------------------------------
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
test_accuracy *= 100
print(f"Final Test Accuracy: {test_accuracy:.2f}%")

# --------------------------------------------------
# 7. Save model
# --------------------------------------------------
model.save('mnist_model_tf.keras')
print("Model saved to mnist_model_tf.keras")

#Listing B-2: JAX Version of Listing 4-1
This code is a JAX version of the simple model shown in Listing 4-1.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, random
import optax
from tensorflow import keras
import numpy as np

# --------------------------------------------------
# 1. Load and prepare data
# --------------------------------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values and apply same normalization as PyTorch version
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = (x_train - 0.1307) / 0.3081
x_test = (x_test - 0.1307) / 0.3081

# Flatten images from 28x28 to 784
x_train = x_train.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)

# Convert to JAX arrays
x_train = jnp.array(x_train)
y_train = jnp.array(y_train)
x_test = jnp.array(x_test)
y_test = jnp.array(y_test)

# --------------------------------------------------
# 2. Define or load model
# --------------------------------------------------
def init_network_params(layer_sizes, key):
    """Initialize network parameters for each layer."""
    params = []
    for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        key, subkey = random.split(key)
        scale = jnp.sqrt(2.0 / (n_in + n_out))
        W = scale * random.normal(subkey, (n_in, n_out))
        b = jnp.zeros(n_out)
        params.append({'W': W, 'b': b})
    return params

def forward(params, x):
    """Forward pass through the network."""
    x = jnp.dot(x, params[0]['W']) + params[0]['b']
    x = jax.nn.relu(x)
    x = jnp.dot(x, params[1]['W']) + params[1]['b']
    x = jax.nn.relu(x)
    x = jnp.dot(x, params[2]['W']) + params[2]['b']
    return x

layer_sizes = [28 * 28, 128, 64, 10]
key = random.PRNGKey(0)
params = init_network_params(layer_sizes, key)

# --------------------------------------------------
# 3. Specify loss function and optimizer
# --------------------------------------------------
def cross_entropy_loss(params, x, y):
    """Compute cross-entropy loss."""
    logits = forward(params, x)
    log_probs = jax.nn.log_softmax(logits)
    return -jnp.mean(log_probs[jnp.arange(len(y)), y])

def accuracy(params, x, y):
    """Compute classification accuracy."""
    logits = forward(params, x)
    preds = jnp.argmax(logits, axis=1)
    return jnp.mean(preds == y)

optimizer = optax.adam(learning_rate=0.001)
opt_state = optimizer.init(params)

@jit
def update(params, opt_state, x, y):
    """Single optimization step."""
    loss, grads = jax.value_and_grad(cross_entropy_loss)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# --------------------------------------------------
# 4. Train model
# --------------------------------------------------
batch_size = 64
n_batches = len(x_train) // batch_size

for epoch in range(5):
    key, subkey = random.split(key)
    perm = random.permutation(subkey, len(x_train))
    x_train_shuffled = x_train[perm]
    y_train_shuffled = y_train[perm]

    running_loss = 0.0

    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = start_idx + batch_size
        x_batch = x_train_shuffled[start_idx:end_idx]
        y_batch = y_train_shuffled[start_idx:end_idx]

        params, opt_state, loss = update(params, opt_state, x_batch, y_batch)
        running_loss += float(loss)

    avg_train_loss = running_loss / n_batches

    # --------------------------------------------------
    # 5. Validate model during training
    # --------------------------------------------------
    val_loss = float(cross_entropy_loss(params, x_test, y_test))
    val_accuracy = float(accuracy(params, x_test, y_test)) * 100

    print(
        f"Epoch {epoch + 1}: "
        f"train loss={avg_train_loss:.4f}, "
        f"val loss={val_loss:.4f}, "
        f"val acc={val_accuracy:.2f}%"
    )

# --------------------------------------------------
# 6. Evaluate and test (final pass)
# --------------------------------------------------
test_accuracy = float(accuracy(params, x_test, y_test)) * 100
print(f"Final Test Accuracy: {test_accuracy:.2f}%")

# --------------------------------------------------
# 7. Save model
# --------------------------------------------------
import pickle
with open('mnist_model_jax.pkl', 'wb') as f:
    params_numpy = jax.tree.map(lambda x: np.array(x), params)
    pickle.dump(params_numpy, f)
print("Model saved to mnist_model_jax.pkl")

#Listing B-3: Exporting a PyTorch Model to ONNX and Running Inference
This example shows how a model trained in PyTorch can be exported to ONNX and then executed using ONNX Runtime.

In [ ]:
# -----------------------------------------------------
# Step 1. Install required packages (for Colab users)
# -----------------------------------------------------
!pip install onnx onnxruntime onnxscript

# -----------------------------------------------------
# Step 2. Import required libraries
# -----------------------------------------------------
import torch
import torch.nn as nn
import onnxruntime as ort
import numpy as np

# -----------------------------------------------------
# Step 3. Define model (same structure as Listing 4-1)
# -----------------------------------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

model = Net()
model.eval()

# -----------------------------------------------------
# Step 4. Create a dummy input
# -----------------------------------------------------
dummy_input = torch.randn(1, 1, 28, 28)

# -----------------------------------------------------
# Step 5. Export model to ONNX
# -----------------------------------------------------
torch.onnx.export(
    model,
    dummy_input,
    "mnist_model.onnx",
    input_names=["input"],
    output_names=["output"]
)

print("Model exported to ONNX format")

# -----------------------------------------------------
# Step 6. Load model with ONNX Runtime
# -----------------------------------------------------
session = ort.InferenceSession("mnist_model.onnx")

# -----------------------------------------------------
# Step 7. Run inference
# -----------------------------------------------------
input_data = np.random.randn(1, 1, 28, 28).astype(np.float32)
inputs = {"input": input_data}

outputs = session.run(None, inputs)

print("ONNX Output:", outputs[0])